# Rakshak AI — Fine-tune Gemma 4 E2B for Offline Disaster Survival (Kaggle)

Fine-tunes **Gemma 4 E2B IT** with Unsloth QLoRA on task data that matches `lib/services/ai_engine.dart` exactly.

**Tasks:** alert→action JSON, distress voice→intent JSON, grounded survival RAG (NDMA/WHO/NDRF), situation brief  
**Languages:** English + Hindi (Devanagari), with code-mixed distress samples  
**Output:** LoRA adapter (`rakshak-lora-final.zip`) to merge and export as `.litertlm` for the app

> Gemma 4 is gated on HuggingFace. Add `HF_TOKEN` in **Kaggle → Add-ons → Secrets**.


## 1) Install dependencies (Kaggle 2x T4 recipe)

In [ ]:
%%capture
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil = f"pillow=={PIL.__version__}"
except Exception:
    _numpy = "numpy"
    _pil = "pillow"

!uv pip install -qqq "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm
!uv pip install -qqq jsonschema
print("Install complete")


## 2) HuggingFace login

In [ ]:
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN") or token
except Exception:
    pass

if token:
    login(token=token)
    print("HF login done")
else:
    print("Add HF_TOKEN to Kaggle Secrets (Gemma 4 is gated).")


## 3) Load base model with Unsloth

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

MODEL_ID = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    finetune_vision_layers=False,
    bias="none",
    random_state=3407,
    use_rslora=False,
    use_gradient_checkpointing="unsloth",
)
print("Model + LoRA ready")


## 4) Load dataset from Kaggle input

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

# This must match the Kaggle dataset slug you upload.
DATA_DIR = Path("/kaggle/input/rakshak-ai-disaster-finetune")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/rakshak-disaster-finetune")
if not DATA_DIR.exists():
    DATA_DIR = Path("./data")  # local fallback

train_path = DATA_DIR / "rakshak_train.jsonl"
eval_path = DATA_DIR / "rakshak_eval.jsonl"

def load_jsonl(p):
    rows = []
    with open(p, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(train_path)
eval_rows = load_jsonl(eval_path)
print("train:", len(train_rows), "eval:", len(eval_rows))

from collections import Counter
print("train tasks:", Counter(r["task"] for r in train_rows))
print("eval tasks:", Counter(r["task"] for r in eval_rows))

assert len(train_rows) >= 300 and len(eval_rows) >= 30, "Dataset split looks wrong"
for r in train_rows + eval_rows:
    for key in ("task", "lang", "system", "user", "assistant"):
        assert r.get(key), f"missing {key} in {r.get('id')}"
    if r["task"] in {"alert", "distress"}:
        json.loads(r["assistant"])  # must be valid JSON
print("dataset checks passed")


## 5) Format as chat messages (match app prompts)

In [ ]:
def to_chat(row):
    # The app merges the system prompt into the first user turn:
    # '$system\n\n$prompt' (see AiEngine._generate). Gemma chat templates
    # do not consistently support a separate system role, so mirror that here.
    system = (row.get("system") or "").strip()
    user = row["user"]
    if system:
        user = f"{system}\n\n{user}"
    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": row["assistant"]},
    ]
    return {"messages": messages}

train_ds = Dataset.from_list([to_chat(r) for r in train_rows])
eval_ds = Dataset.from_list([to_chat(r) for r in eval_rows])

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = train_ds.map(format_chat)
eval_ds = eval_ds.map(format_chat)
print(train_ds[0]["text"][:300])


## 6) Train with SFT

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    output_dir="./rakshak-lora-out",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=args,
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

trainer.train()


## 7) Save the LoRA adapter

In [ ]:
import os, shutil
out = "./rakshak-lora-final"
os.makedirs(out, exist_ok=True)
model.save_pretrained(out)
tokenizer.save_pretrained(out)
shutil.make_archive("rakshak-lora-final", "zip", out)
print("Saved + zipped", out)


## 8) Quick held-out evaluation

In [ ]:
import json, re
from jsonschema import validate
from unsloth import FastModel

ALERT_SCHEMA = {
    "type": "object",
    "required": ["hazard", "severity", "actions", "donts", "speak"],
    "properties": {
        "hazard": {"type": "string"},
        "severity": {"enum": ["red", "orange", "yellow", "green"]},
        "actions": {"type": "array", "items": {"type": "string"}, "minItems": 1, "maxItems": 4},
        "donts": {"type": "array", "items": {"type": "string"}, "maxItems": 3},
        "speak": {"type": "string"},
    },
}
DISTRESS_SCHEMA = {
    "type": "object",
    "required": ["intent", "injured", "trapped", "people", "landmark", "summary"],
    "properties": {
        "intent": {"enum": ["beacon_safe", "beacon_help", "beacon_evac", "open_survival", "unknown"]},
        "injured": {"type": "boolean"},
        "trapped": {"type": "boolean"},
        "people": {"type": "integer"},
        "landmark": {"type": ["string", "null"]},
        "summary": {"type": "string"},
    },
}
INTENTS = ["beacon_safe", "beacon_help", "beacon_evac", "open_survival", "unknown"]

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

FastModel.for_inference(model)

alert_ok = distress_ok = intent_ok = 0
alert_n = distress_n = 0

for r in eval_rows:
    if r["task"] == "alert" and alert_n < 10:
        messages = [{"role": "user", "content": r["system"] + "\n\n" + r["user"]}]
        inp = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to(model.device)
        out = model.generate(input_ids=inp, max_new_tokens=256, do_sample=False)
        resp = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
        j = extract_json(resp)
        if j:
            try:
                validate(j, ALERT_SCHEMA)
                alert_ok += 1
            except Exception:
                pass
        alert_n += 1
    elif r["task"] == "distress" and distress_n < 10:
        messages = [{"role": "user", "content": r["system"] + "\n\n" + r["user"]}]
        inp = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to(model.device)
        out = model.generate(input_ids=inp, max_new_tokens=160, do_sample=False)
        resp = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
        j = extract_json(resp)
        if j:
            try:
                validate(j, DISTRESS_SCHEMA)
                if j.get("intent") in INTENTS:
                    intent_ok += 1
                distress_ok += 1
            except Exception:
                pass
        distress_n += 1

print("alert json valid:", alert_ok, "/", alert_n)
print("distress json valid:", distress_ok, "/", distress_n, " intent_ok:", intent_ok)


## 9) Next steps

1. Download `rakshak-lora-final.zip` from Kaggle Outputs.
2. In repo root: `unzip rakshak-lora-final.zip -d rakshak-lora-final/`
3. Run `python fine-tuning/convert_to_litertlm.py` to merge + export `.litertlm`.
4. Push: `adb push rakshak-output/rakshak-merged-litertlm/gemma-4-E2B-it.litertlm /sdcard/Download/`
5. Relaunch the app; `AiEngine._installFromDevice` will import it.
